--- 
## 🧩 Paso 1: Cargar y explorar

Antes de limpiar o combinar los datos, es necesario **familiarizarte con la estructura de los tres datasets**.  
En esta etapa, validarás que los archivos se carguen correctamente, conocerás sus columnas y tipos de datos, y detectarás posibles inconsistencias.

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:**  
Tener los **3 datasets listos en memoria**, entender su contenido y realizar una revisión preliminar.

**Instrucciones:**  
- Importa las librerías necesarias (por ejemplo `pandas`, `seaborn`, `matplotlib.pyplot`)
- Carga los archivos CSV usando `pd.read_csv()`:
  - **`/datasets/plans.csv`**  
  - **`/datasets/users_latam.csv`**  
  - **`/datasets/usage.csv`**  
- Guarda los DataFrames en las variables: `plans`, `users`, `usage`.  
- Muestra las primeras filas de cada DataFrame usando `.head()`.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
# cargar archivos
plans = pd.read_csv('/datasets/plans.csv')
users = pd.read_csv('/datasets/users_latam.csv')
usage = pd.read_csv('/datasets/usage.csv')

In [ ]:
# mostrar las primeras 5 filas de plans
plans.head(5)


In [ ]:
# mostrar las primeras 5 filas de users
users.head(5)

In [ ]:
# mostrar las primeras 5 filas de usage
usage.head(5)

**Tip:** Si no usas `print()` la tabla se vera mejor.

### 1.2 Exploración de la estructura de los datasets

**🎯 Objetivo:**  
Conocer la **estructura de cada dataset**, revisar cuántas filas y columnas tienen, identificar los **tipos de datos** de cada columna y detectar posibles **inconsistencias o valores nulos** antes de iniciar el análisis.

**Instrucciones:**  
- Revisa el **número de filas y columnas** de cada dataset usando `.shape`.  
- Usa `.info()` en cada DataFrame para obtener un **resumen completo** de columnas, tipos de datos y valores no nulos.  

In [ ]:
# revisar el número de filas y columnas de cada dataset
print("plans", plans.shape)
print("users",users.shape)
print("usage",usage.shape)

In [ ]:
# inspección de plans con .info()
print("plans",plans.shape)
plans.info()

In [ ]:
# inspección de users con .info()
print("users",users.shape)
users.info()

In [ ]:
usage.head()

In [ ]:
# inspección de usage con .info()
print("usage",usage.shape)
usage.info()

---

## 🧩Paso 2: Identificación de problemas de calidad de datos

### 2.1 Revisión de valores nulos

**🎯 Objetivo:**  
Detectar la presencia y magnitud de valores faltantes para evaluar si afectan el análisis o requieren imputación/eliminación.

**Instrucciones:**  
- Cuenta valores nulos por columna para cada dataset.
- Calcula la proporción de nulos por columna para cada dataset.

El dataset `plans` solamente tiene 2 renglones y se puede observar que no tiene ausentes, por ello no necesita exploración adicional.

<br>
<details>
<summary>Haz clic para ver la pista</summary>
Usa `.isna().sum()` para contar valores nulos y usa `.isna().mean()` para calcular la proporción.

In [ ]:
# cantidad de nulos para users
print(users.isna().sum())



In [ ]:
# cantidad de nulos para usage
print(usage.isna().sum())

✍️ **Comentario**: Haz doble clic en este bloque y escribe tu diagnóstico al final del bloque. Incluye qué ves y que acción recomendarías para cada caso.

💡 **Nota:** Justifica tus decisiones brevemente (1 línea por caso).
* Hint:
 - Si una columna tiene **más del 80–90% de nulos**, normalmente se **ignora o elimina**.  
 - Si tiene **entre 5% y 30%**, generalmente se **investiga para imputar o dejar como nulos**.  
 - Si es **menor al 5%**, suele ser un caso simple de imputación o dejar como nulos. 
 
 ---

**Valores nulos**  
- ¿Qué columnas tienen valores faltantes y en qué proporción? churn date tiene 3534 de 4000 son la mayoria y city son 469 de 3500 se debe investigar para imputar esos valores
- Indica qué harías: ¿imputar, eliminar, ignorar?
- en churn date practicamente es el 87.5% de los datos se debe eliminar para que no afecte a las predicciones.
- ignorar en city ya que es una parte muy baja lo que tiene como valores faltantes.

### 2.2 Detección de valores inválidos y sentinels

🎯 **Objetivo:**  
Identificar sentinels: valores que no deberían estar en el dataset.

**Instrucciones:**
- Explora las columnas numéricas con **un resumen estadístico** y describe brevemente que encontraste.
- Explora las columnas categóricas **relevantes**, revisando sus valores únicos y describe brevemente que encontraste.


El dataset `plans` solamente tiene 2 renglones, por ello no necesita exploración adicional.

In [ ]:
# explorar columnas numéricas de users
users.info()
users.describe()

- La columna `user_id` ... Haz doble clic en este bloque y escribe qué ves.tiene unos valores de 4000 y minimo de -999 esa edad no existe. pero se tendria que hacer un histograma para ver si son reales o son error de dedo.


In [ ]:
# explorar columnas numéricas de usage
usage.describe()

- Las columnas `id` y `user_id`...Haz doble clic en este bloque y escribe qué ves.
- Las columnas ...en ID se ve bien media, al 50, al 25 y todas las demas si cuadran pero en User ID esta todo erroneo no coincide la mitad, ni ninguno de los rangos.

In [ ]:
# explorar columnas categóricas de users
columnas_user = ['city', 'plan']


- La columna `city` ...
- La columna `plan` ...

In [ ]:
# explorar columna categórica de usage
usage['type'] # completa el código

- La columna `type` ...dice llamadas y texto se me hace raro que tenga de ambos si es llamada debe tener solo llamadas los mensajes de texto deben estar en otro rango o categoria.


---
✍️ **Comentario**: Haz doble clic en este bloque y escribe tu diagnóstico. Incluye qué ves y que acción recomendarías para cada caso. 

**Valores inválidos o sentinels**  
- ¿En qué columnas encontraste valores inválidos o sentinels?en la 1 y 2 dice texto
- ¿Qué acción tomarías?
- reemplazar centinels, convertir fechas, marcar NA segun reglas.

### 2.3 Revisión y estandarización de fechas

**🎯 Objetivo:**  
Asegurar que las columnas de fecha estén correctamente formateadas y detectar años fuera de rango que indiquen errores de captura.

**Instrucciones:**  
- Convierte las columnas de fecha a tipo fecha y asegurate de que el código sea a prueba de errores.  
- Revisa cuántas veces aparece cada año.
- Identifica fechas imposibles (ej. años futuros o negativos).

Toma en cuenta que tenemos datos registrados hasta el año 2024.

In [ ]:
# Convertir a fecha la columna `reg_date` de users
users['reg_date'] = pd.to_datetime(users['reg_date'], errors='coerce')

In [ ]:
# Convertir a fecha la columna `date` de usage
usage['date'] =pd.to_datetime(usage['date'],errors='coerce')

In [ ]:
# Revisar los años presentes en `reg_date` de users
users.reg_date.dt.year.value_counts()

En `reg_date`, ... haz doble clic en este bloque y escribe qué ves existen varios años para analizar como son 2022, 2023, 2024 y 2026 falta 2025 sin saber cual es la razón.

In [ ]:
# Revisar los años presentes en `date` de usage
usage.date.dt.year.value_counts()

En `date`, ... haz doble clic en este bloque y escribe qué ves.  
Basaremos el análisis en estas fechas. en usage solo esta el 2024 seria muy desigual una diferencia de años descomunal.

✍️ **Comentario**: escribe tu diagnóstico, e incluye **qué acción recomendarías** para cada caso:

**Fechas fuera de rango**  
- ¿Aparecen años imposibles? (años muy viejos o sin transcurrir al momento de guardar los datos)en un df hay varios años del 22 al 26 y en el otro solo existe el 24 
- ¿Qué harías con ellas? solo se puede trabajar con el 24 si no hay otra forma de averiguar como o donde se encuentran los años faltantes y porque.

---

## 🧩Paso 3: Limpieza básica de datos

### 3.1 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Aplicar reglas de limpieza para reemplazar valores sentinels y corregir fechas imposibles.

**Instrucciones:**  
- En `age`, reemplaza el sentinel **-999** con la mediana.
- En `city`, reemplaza el sentinel `"?"` por valores nulos (`pd.NA`).  
- Marca como nulas (`pd.NA`) las fechas fuera de rango.

In [ ]:
# Reemplazar -999 por la mediana de age
age_median = users.age.median()
users["age"]=users["age"].replace(-999,age_median)

# Verificar cambios
users['age'].describe()

In [ ]:
# Reemplazar ? por NA en city
users["city"]=users["city"].replace("?",pd.NA)
users["city"].value_counts()


# Verificar cambios


In [ ]:
# Marcar fechas futuras como NA para reg_date
users['year']= users['reg_date'].dt.year
users["reg_date"] = np.where(users["year"] > 2024, pd.NaT,users['reg_date'])
users.info()

### 3.2 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Decidir qué hacer con los valores nulos según su proporción y relevancia.

**Instrucciones:**
- Verifica si los nulos en `duration` y `length` son **MAR**(Missing At Random) revisando si dependen de la columna `type`.  
  Si confirmas que son MAR, **déjalos como nulos** y justifica la decisión.
 se deben dejar como nulos porque duration deben ser valores nulos debido a que no puede haber texto en una llamada y en length no puede haber duracion de llamada porque son puro texto.

In [ ]:
# Verificación MAR en usage (Missing At Random) para duration
print(usage['duration'].isna().groupby(usage['type']).sum())
print(usage['length'].isna().groupby(usage['type']).sum())

usage.head(5)



Haz doble clic aquíy escribe que tu diagnostico de nulos en `duration` y `length`

---

## 🧩Paso 4: Summary statistics de uso por usuario


### 4.1 Agrupación por comportamiento de uso

🎯**Objetivo**: Resumir las variables clave de la tabla `usage` **por usuario**, creando métricas que representen su comportamiento real de uso histórico. 

**Instrucciones:**: 
1. Construye una tabla agregada de `usage` por `user_id` que incluya:
- número total de mensajes  
- número total de llamadas  
- total de minutos de llamadas

2. Renombra las columnas para que tengan nombres claros:  
- `cant_mensajes`  
- `cant_llamadas`  
- `cant_minutos_llamada`
3. Combina esta tabla con `users`.

In [ ]:
# Columnas auxiliares
usage["is_text"] = (usage["type"] == "text").astype(int) #conocer el total de mensajes
usage["is_call"] = (usage["type"] == "call").astype(int) #conocer el total de llamadas


# Agrupar información por usuario

usage_agg = usage.groupby('user_id').agg(
    cant_mensajes=('is_text', 'sum'),
    cant_llamadas=('is_call','sum'),
    cant_minutos_llamada=('duration', 'sum')
).reset_index()
# observar resultado
usage_agg.head(3)

In [ ]:
# Combinar la tabla agregada con el dataset de usuarios
user_profile = pd.merge(usage_agg,users,on="user_id",how="left")
user_profile.head(5)

### 4.2 4.2 Resumen estadístico por usuario durante el 2024

🎯 **Objetivo:** Analizar las columnas numéricas y categóricas de los usuarios, para identificar rangos, valores extremos y distribución de los datos antes de continuar con el análisis.

**Instrucciones:**  
1. Para las columnas **numéricas** relevantes, obtén un resumen estadístico (media, mediana, mínimo, máximo, etc.).  
2. Para la columna **categórica** `plan`, revisa la distribución en **porcentajes** de cada categoría.

In [ ]:
# Resumen estadístico de las columnas numéricas
user_profile.describe()

In [ ]:
user_profile.describe(include=["object"])

In [ ]:
# Distribución porcentual del tipo de 
# Seleccionar las columnas categóricas
categoricas = user_profile.select_dtypes(include=['object', 'category', 'string']).columns

# Mostrar el porcentaje de cada categoría en cada columna
for col in categoricas:
    print(f"\nDistribución porcentual de '{col}':")
    print(((user_profile[col].value_counts(normalize=True) * 100).round(2)).astype(str) + "%")

---

## 🧩Paso 5: Visualización de distribuciones (uso y clientes) y outliers


### 5.1 Visualización de Distribuciones

🎯 **Objetivo:**  
Entender visualmente cómo se comportan las variables clave tanto de **uso** como de **clientes**, observar si existen diferencias según el tipo de plan, y analizar la **forma de la distribución**.

**Instrucciones:**  
Graficar **histogramas** para las siguientes columnas:  
- `age` (edad de los usuarios)
- `cant_mensajes`
- `cant_llamadas`
- `total_minutos_llamada` 

Después de cada gráfico, escribe un **insight** respecto al plan y la variable, por ejemplo:  
- "Dentro del plan Premium, hay mayor proporción de usuarios que ocupan más las llamadas en ciertos rangos de edades a los 50 hay pico y entre los 60 y 75 de edad .."  
- "Los usuarios Básico tienden a hacer muchas llamadas y enviar algunos mensajes."  existe algún patrón debido a que tienen que recargar saldo en lugar de estar con plan."
- ¿Qué tipo de distribución tiene ? (simétrica, sesgada a la derecha o a la izquierda) ocupan entre 3 y 8 mensajes para el tipo de contrato con el que cuentan esta sesgada a la izquierda.

**Hint**  
Para cada histograma, 
- Usa `hue='plan'` para ver cómo varían las distribuciones según el plan (Básico o Premium).
- Usa `palette=['skyblue','green']`
- Agrega título y etiquetas

In [ ]:
# Histograma para visualizar la edad (age)

sns.histplot(data=user_profile, x='age', hue='plan', palette=['skyblue','green'])
plt.title('Histograma de Edad')
plt.xlabel('age')
plt.show()




💡Insights: 
- Distribución ...


In [ ]:
user_profile.columns

In [ ]:
# Histograma para visualizar la cant_mensajes
sns.histplot(data=user_profile, x='cant_mensajes', hue='plan', palette=['skyblue','green'])
plt.title('Histograma de Cantidad de mensajes')
plt.xlabel('cant_mensajes')
plt.show()



💡Insights: 
- ....

In [ ]:
# Histograma para visualizar la cant_llamadas
sns.histplot(data=user_profile, x='cant_llamadas', hue='plan', palette=['skyblue','green'])

💡Insights: 
- Distribución ...

In [ ]:
# Histograma para visualizar la cant_minutos_llamada
sns.histplot(data=user_profile, x='cant_minutos_llamada', hue='plan', palette=['skyblue','green'])

💡Insights: 
- ...

### 5.2 Identificación de Outliers

🎯 **Objetivo:**  
Detectar valores extremos en las variables clave de **uso** y **clientes** que podrían afectar el análisis, y decidir si requieren limpieza o revisión adicional.

**Instrucciones:**  
- Usa **boxplots** para identificar visualmente outliers en las siguientes columnas:  
  - `age` 
  - `cant_mensajes`
  - `cant_llamadas`
  - `total_minutos_llamada`  
- Crea un **for** para generar los 4 boxplots automáticamente.
<br>

- Después de crear los gráfico, responde si **existen o no outliers** en las variables.  
- Si hay outliers, crea otro bucle para calcular los límites de esas columnas usando el **método IQR** y decide qué hacer con ellos.
  - Si solamente hay outliers de un solo lado, no es necesario calcular ambos límites.

**Hint:**
- Dentro del bucle, usa `plt.title(f'Boxplot: {col}')` para que el título cambie acorde a la columna.

In [ ]:
# Visualizando usando BoxPlot 
columnas_numericas = ['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

for col in columnas_numericas:
    user_profile.boxplot(column=col)
    plt.title(f'Boxplot: {col}')
    plt.show()

💡Insights: 
- Age: ...(presenta o no outliers) si
- cant_mensajes: ...si 17.5 fuera de rango
- cant_llamadas: ... si 15 fuera de rango
- cant_minutos_llamada: ...se alcanza a apreciar hasta 150 fuera de rango es el histograma con más outliers que tiene el data frame.

In [ ]:
# Calcular límites con el método IQR
columnas_limites = ["age", "cant_llamadas", "cant_mensajes", "cant_minutos_llamada"]



In [ ]:
# Revisa los limites superiores y el max, para tomar la decisión de mantener los outliers o no
user_profile[columnas_limites].describe()

💡Insights: 
- cant_mensajes: mantener o no outliers, porqué? no 
- cant_llamadas: mantener o no outliers, porqué? no
- cant_minutos_llamada: mantener o no outliers, porqué? si hay que quitarlos para centrar la dispersion de datos.

---

## 🧩Paso 6: Segmentación de Clientes

### 6.1 Segmentación de Clientes Por Uso

🎯 **Objetivo:** Clasificar a cada usuario en un grupo de uso (Bajo uso, Uso medio, Alto uso) basándose en la cantidad de llamadas y mensajes registrados.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_uso` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones de llamadas y mensajes y asigna:
  - `'Bajo uso'` cuando llamadas < 5 y mensajes < 5
  - `'Uso medio'` cuando llamadas < 10 y mensajes < 10
  - `'Alto uso'` para el resto de casos

In [ ]:
# Crear columna grupo_uso
condiciones = [
    (user_profile["cant_llamadas"] < 5) & (user_profile['cant_mensajes'] <5),
    (user_profile["cant_llamadas"] < 10) & (user_profile['cant_mensajes'] < 10)
]
resultados = ['bajo uso', 'uso medio']

user_profile['grupo_uso'] = np.select(condiciones, resultados, default='alto uso')



In [ ]:
# verificar cambios
user_profile.head()

### 6.2 Segmentación de Clientes Por Edad

🎯 **Objetivo:**: Clasificar a cada usuario en un grupo por **edad**.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_edad` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones y asigna:
  - `'Joven'` cuando age < 30
  - `'Adulto'` cuando age < 60
  - `'Adulto Mayor'` para el resto de casos

In [ ]:
# Crear columna grupo_edad
condiciones = [
    user_profile["age"] < 30, user_profile['age'] <60
]
resultados = ['joven', 'adulto']

user_profile['grupo_edad'] = np.select(condiciones, resultados, default='adulto mayor')

In [ ]:
# verificar cambios
user_profile.head()

### 6.3 Visualización de la Segmentación de Clientes

🎯 **Objetivo:** Visualizar la distribución de los usuarios según los grupos creados: **grupo_uso** y **grupo_edad**.

**Instrucciones:**  
- Crea dos gráficos para las variables categóricas `grupo_uso` y `grupo_edad`.
- Agrega título y etiquetas a los ejes en cada gráfico.

In [ ]:
# Visualización de los segmentos por uso
sns.countplot(data=user_profile, x = 'grupo_uso')

plt.title("tipo de uso")
plt.ylabel("total")
plt.show()

In [ ]:
# Visualización de los segmentos por edad
sns.countplot(data=user_profile, x = 'grupo_edad')
plt.title("rango de edad")
plt.ylabel("total")
plt.show()


---
## 🧩Paso 7: Insight Ejecutivo para Stakeholders

🎯 **Objetivo:** Traducir los hallazgos del análisis en conclusiones accionables para el negocio, enfocadas en segmentación, patrones de uso y oportunidades comerciales.

**Preguntas a responder:** 
- ¿Qué problemas tenían originalmemte los datos?¿Qué porcentaje, o cantidad de filas, de esa columna representaban? había celdas con numeros muy dispersos, que ocasionaban confusion a la hora de analizar, como edades de -99 o fechas erroneas, espacios que hacian no considerar algunos elementos por error de dedo.

- ¿Qué segmentos de clientes identificaste y cómo se comportan según su edad y nivel de uso?   los adultos mayores según grafico se la pasan hablando por telefono todo el tiempo, los adultos media edad son 2500 usuarios representan a la mayoria y los jovenes son casi 800 usuarios pero son mas de mensajes. 
- ¿Qué segmentos parecen más valiosos para ConnectaTel y por qué?

- la mayoria de los clientes pertenecen al grupo de uso medio, una cantidad minima de cientes pertenece a usuarios de alto uso, los el grupo de bajo uso representa una oportunidad para incrementar el consumo en paquete.

- los que mas usan el celular son el grupo mas pequeño pero son los que generan mas ganancia
- ¿Qué patrones de uso extremo (outliers) encontraste y qué implican para el negocio?

- hay una proporcion de clientes que consumen mucho por lo que los ingresos dependen principalmente de usuarios de uso medio 

el comportamiento no sigue una distribucion normal estandar; hay una retencion masiva en niveles de consumo moderados, hay una brecha considerable entre uso medio y alto uso que debe investigarse si hay que crear la necesidad, precio alto, o desconocimiento de ventajas en los planes de renta.
- ¿Qué recomendaciones harías para mejorar la oferta actual de planes o crear nuevos planes basados en los segmentos y patrones detectados?
-
 es importante fidelizar a los usuarios de alto uso, ya que suelen aportar un mayor ingreso a la empresa. sugiero implementar campañas de reactivacion de lineas, promociones o planes ajustados a sus necesidades ofrecer servicios adicionales como:
- mayor capacidad de datos
- streaming de los mas populares como son netflix, HBO max, Disney plus, paramount pictures entre otras.
- planes para toda la familia que los haga quedarse en la empresa
- GPS kids para saber donde estan sus hijos
- equipos a credito el nuevo i phone en paquete con smart swatch para que no quieran irse a otra empresa.
- seguro para dispositivos por si te asaltan, o se daña el equipo.


✍️ **Escribe aquí tu análisis ejecutivo:**

### Análisis ejecutivo

⚠️ **Problemas detectados en los datos**
- habia datos muy raros con fechas dispersas
- espacios dobles y sin orden algunos datos


🔍 **Segmentos por Edad**
- edades de -99 que hacian complicado hacer el analisis
- las edades que mas usan el equipo son los que tienen mayor edad 


📊 **Segmentos por Nivel de Uso**
- la mayoria de los clientes pertenece al grupo de uso medio
- existe una cantidad minima de usuarios de alto uso.


➡️ Esto sugiere que ...


💡 **Recomendaciones**
-  es importante comprometer a los usuarios de alto uso, por ser los mas valiosos ya que suelen aportar un mayor ingreso a la empresa. sugiero implementar campañas de reactivacion de lineas, promociones o planes ajustados a sus necesidades ofrecer servicios adicionales como:
- mayor capacidad de datos
- streaming de los mas populares como son netflix, HBO max, Disney plus, paramount pictures entre otras.
- planes para toda la familia que los haga quedarse en la empresa
- GPS kids para saber donde estan sus hijos
- equipos a credito el nuevo i phone en paquete con smart swatch para que no quieran irse a otra empresa.
- seguro para dispositivos por si te asaltan, o se daña el equipo.
y una oferta de pagar la mitad del plan si te cambias de compañia.
- implementar campañas para reactivacion para clientes de bajo uso con promociones temporales, bonos de datos dobles, descuentos en cines o de acuerdo a su rango de edad y uso.
- fidelizar a los clientes de alto uso con programas VIP beneficios exclusivos y atencion preferencial para reducir el riesgo de cambio de compañia. 

---

## 🧩Paso 8 Cargar tu notebook y README a GitHub

🎯 **Objetivo:**  
Entregar tu análisis de forma **profesional**, **documentada** y **versionada**, asegurando que cualquier persona pueda revisar, ejecutar y entender tu trabajo.



### Opción A : Subir archivos desde la interfaz de GitHub (UI)

1. Descarga este notebook (`File → Download .ipynb`).  
2. Entra a tu repositorio en GitHub (por ejemplo `telecom-analysis` o `sprint7-final-project`).  
3. Sube tu notebook **Add file → Upload files**.  

---

### Opción B : Guardar directo desde Google Colab

1. Abre tu notebook en Colab.  
2. Ve a **File → Save a copy in GitHub**.  
3. Selecciona el repositorio y la carpeta correcta (ej: `notebooks/`).  
4. Escribe un mensaje de commit claro, por ejemplo:  
    - `feat: add final ConnectaTel analysis`
    - `agregar version final: Análisis ConnectaTel`
5. Verifica en GitHub que el archivo quedó en el lugar correcto y que el historial de commits se mantenga limpio.

---

Agrega un archivo `README.md` que describa de forma clara:
- el objetivo del proyecto,  
- los datasets utilizados,  
- las etapas del análisis realizadas,  
- cómo ejecutar el notebook (por ejemplo, abrirlo en Google Colab),  
- una breve guía de reproducción.
---

Link a repositorio público del proyecto: `LINK a tu repo aquí`